# X-Bagger Finder & Forward Screener

Four-stage pipeline:
1. **Historical scan** (yfinance) — find every ticker that grew X-times within 5y/10y/20y rolling windows.
2. **Winner deep-dive** (LLM + web search + SEC filings) — extract business model, moat, inflection point, and macro context for each historical winner.
3. **Pattern extraction** (LLM consolidation) — synthesize the winners into a single "pattern fingerprint" describing what they had in common.
4. **Forward screener** (rule-based pre-filter + LLM scoring) — score the current SEC universe against the pattern, surface the best matches.

> **Survivorship bias:** This pipeline scans only currently-listed SEC tickers. Historical multi-baggers that delisted (acquired, taken private, went to zero) are excluded. We are explicitly hunting for surviving winners, but be aware that the extracted pattern over-weights traits common to companies that *kept* trading.

## Setup

In [ ]:
import sys, os, json
import pandas as pd
sys.path.append("../")

from wallstreet_quant.x_bagger import XBaggerScanner

assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY"
assert os.getenv("edgar_identity"), "Set edgar_identity to 'Your Name email@example.com'"

## Load Stock Universe

Use the SEC company tickers JSON (~10k currently-listed companies).

In [ ]:
with open("company_tickers.json") as f:
    sec_data = json.load(f)

tickers = [v['ticker'] for v in sec_data.values()]
print(f"Universe: {len(tickers)} SEC-listed tickers")

## Initialize Scanner

In [ ]:
scanner = XBaggerScanner(
    batch_delay=0.1,                          # seconds between yfinance calls
    model="gpt-5.2-pro",                      # default for per-stock analyses
    consolidation_model="o3",                 # for the pattern-extraction call
    cache_dir="./output/x_bagger_cache",      # disk cache for Stage 2 LLM results
)

## Stage 1: Historical Scan

For each ticker, download max-period yfinance history and compute the maximum drawup multiple within rolling 5y / 10y / 20y windows. **This is the slow stage** — expect 60-90 minutes for the full SEC universe.

Stage 1 results are saved to CSV so you can re-run downstream stages without repeating the scan.

### Quality filters

Without filtering you'll get penny-stock noise — names like TMGI showing "100,000x" because they went $0.0001 → $0.01. Two filters suppress this:

- **`min_trough_price`** (default `$1`): the trough close must be at least this much. Kills sub-penny pump-and-dumps.
- **`min_trough_dollar_volume`** (default `$1M`): 30-day average dollar volume around the trough must be at least this much. Kills illiquid junk that nobody can actually buy.

If you already have a Stage 1 CSV with garbage in it, use `scanner.revalidate_stage1(stage1_df, ...)` to re-run the math on just those tickers with stricter filters — much faster than re-scanning the whole universe.

In [ ]:
stage1_path = "./output/x_bagger_stage1.csv"
os.makedirs(os.path.dirname(stage1_path), exist_ok=True)

if os.path.exists(stage1_path):
    print(f"Loading cached Stage 1 results from {stage1_path}")
    stage1 = pd.read_csv(stage1_path)
else:
    stage1 = scanner.find_historical_winners(
        tickers,
        windows=[5, 10, 20],
        min_trough_price=1.0,           # drop sub-$1 troughs
        min_trough_dollar_volume=1e6,   # drop trough days with <$1M avg dollar volume
    )
    stage1.to_csv(stage1_path, index=False)
    print(f"Stage 1 saved to {stage1_path}")

print(f"Tickers with usable history: {len(stage1)}")
print("\nTop 20 by 10-year drawup:")
stage1.nlargest(20, 'max_10y')[['ticker','current_price','max_5y','max_10y','max_20y','trough_price_10y','trough_date_10y','peak_date_10y']]

### Revalidate an existing Stage 1 CSV

If your existing CSV has penny-stock junk (e.g. TMGI showing 100,000x), re-run the math with stricter filters on just those tickers — no need to rescan the full universe.

In [ ]:
# Clean an existing Stage 1 CSV in place. Loads from disk if `stage1` not in memory.
if 'stage1' not in dir() or stage1 is None:
    stage1 = pd.read_csv(stage1_path)
    print(f"Loaded {len(stage1)} rows from {stage1_path}")

stage1 = scanner.revalidate_stage1(
    stage1,
    min_trough_price=1.0,
    min_trough_dollar_volume=1e6,
    windows=[5, 10, 20],
)
stage1.to_csv(stage1_path, index=False)
print(f"Revalidated. {len(stage1)} tickers, top 20 by 10y multiple:")
stage1.nlargest(20, 'max_10y')[['ticker','current_price','max_10y','trough_price_10y','trough_date_10y','peak_date_10y']]

## Filter Winners

Pick a `(window, multiple)` gate. Common choices:

| Threshold | Description |
|---|---|
| 10x in 5y | Very recent rapid runs (post-2020 momentum names) |
| 100x in 10y | Classic decade multi-bagger (NVDA, TSLA, MNST, etc.) |
| 1000x in 20y | Generational wealth-builders (very few qualify) |

In [ ]:
WINDOW_YEARS = 10
MIN_MULTIPLE = 100

winners = scanner.filter_winners(stage1, window_years=WINDOW_YEARS, min_multiple=MIN_MULTIPLE)
print(f"{len(winners)} tickers cleared {MIN_MULTIPLE}x in {WINDOW_YEARS}y")
winners[['ticker', f'max_{WINDOW_YEARS}y', f'trough_date_{WINDOW_YEARS}y', f'peak_date_{WINDOW_YEARS}y']].head(30)

## Stage 2: Winner Deep-Dive

For each winner, run three LLM analyses:
1. **Business model & moat** (web search) — `WinnerBusinessModel`
2. **Inflection-point analysis** (SEC filing near the trough) — `WinnerInflection`
3. **Macro / sector context** (web search) — `WinnerMarketContext`

Results are cached to `./output/x_bagger_cache/<ticker>.pkl` so re-runs are free.

In [ ]:
profiles = scanner.analyze_winners(winners, window_years=WINDOW_YEARS)
print(f"Analysed {len(profiles)} winners.")

# Quick look at the first profile
if profiles:
    p = profiles[0]
    print(f"\n=== {p['ticker']} (max {p.get(f'max_{WINDOW_YEARS}y'):.0f}x) ===")
    print(f"Run: {p.get('trough_date')} → {p.get('peak_date')}")
    bm = p.get('business_model', {})
    print(f"\nBusiness model: {bm.get('business_model')}")
    print(f"Moat:           {bm.get('moat')}")
    print(f"Sector:         {bm.get('sector')} / {bm.get('industry')}")
    inf = p.get('inflection', {})
    print(f"\nKey catalyst:   {inf.get('key_catalyst')}")
    print(f"Financial sig.: {inf.get('financial_signature')}")
    print(f"Pre-run cap:    {inf.get('pre_run_market_cap_estimate')}")
    mc = p.get('market_context', {})
    print(f"\nMacro tailwinds:  {mc.get('macro_tailwinds')}")
    print(f"Sector tailwinds: {mc.get('sector_tailwinds')}")

## Stage 3: Pattern Extraction

Single LLM consolidation call (`o3` by default) that synthesizes all winner profiles into a `WinnerPatterns` fingerprint.

In [ ]:
patterns = scanner.extract_patterns(profiles)

print("=== Common Sectors ===")
for s in patterns.common_sectors: print(f"  - {s}")
print("\n=== Common Moats ===")
for m in patterns.common_moats: print(f"  - {m}")
print("\n=== Common Inflection Signatures ===")
for i in patterns.common_inflection_signatures: print(f"  - {i}")
print("\n=== Common Pre-Run Traits ===")
for t in patterns.common_pre_run_traits: print(f"  - {t}")
print("\n=== Common Macro Themes ===")
for m in patterns.common_macro_themes: print(f"  - {m}")
print("\n=== Meta Pattern ===")
print(patterns.meta_pattern)

## Stage 4: Forward Screener

Two sub-stages:
1. **Pre-filter** the current universe with cheap yfinance `.info` calls (drop mega-caps and irrelevant sectors).
2. **LLM-score** each survivor against the pattern fingerprint via web search.

Tune `max_market_cap` and `sector_whitelist` based on the `common_pre_run_traits` and `common_sectors` from Stage 3.

In [ ]:
# Tune these from the Stage 3 patterns output
MAX_MARKET_CAP = 5e9              # USD — drop anything bigger than this
SECTOR_WHITELIST = None           # e.g. ["Technology", "Healthcare"] — None = all sectors

candidates = scanner.score_current_stocks(
    tickers,
    patterns,
    max_market_cap=MAX_MARKET_CAP,
    sector_whitelist=SECTOR_WHITELIST,
)

print(f"Scored {len(candidates)} candidates.")
candidates.head(50)[['ticker','sector','market_cap','score','matched_traits','missing_traits']]

## Save All Stage Outputs

In [ ]:
results = {
    "stage1": stage1,
    "winners": winners,
    "profiles": profiles,
    "patterns": patterns,
    "candidates": candidates,
}
filepath = scanner.save_results(results, output_dir=".")
print(f"Saved to: {filepath}")

## Debug: Single-Stock Inspection

Walk one ticker through every stage to inspect intermediate outputs.

In [ ]:
test_ticker = "NVDA"

# Stage 1: historical drawup
import yfinance as yf, numpy as np
df = yf.download(test_ticker, period="max", progress=False, auto_adjust=True)
if isinstance(df.columns, pd.MultiIndex):
    df.columns = [c[0] for c in df.columns]
closes = df["Close"].values.astype(float)
for y in [5, 10, 20]:
    res = XBaggerScanner._max_drawup(closes, df.index, y * 252)
    print(f"  {y}y: {res['multiple']:.1f}x  ({res['trough_date']} → {res['peak_date']})")

# Stage 2: deep-dive
row = stage1[stage1['ticker'] == test_ticker].iloc[0]
profile = scanner.analyze_winner(test_ticker,
                                  trough_date=row[f'trough_date_{WINDOW_YEARS}y'],
                                  peak_date=row[f'peak_date_{WINDOW_YEARS}y'])
print("\n=== Profile ===")
print(json.dumps(profile, indent=2, default=str)[:2000])